In [26]:
import os
import pandas as pd
import anndata as ad

# -----------------------------
# Input / Output paths
# -----------------------------
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain/"
OUTPUT_FOLDER     = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
COMMON_MZS_CSV    = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered.h5ad",
    "halfbrain_yc_2_filtered.h5ad",
    "halfbrain_yc_3_filtered.h5ad",
    "halfbrain_yc_4_filtered.h5ad",
    "halfbrain_yad_1_filtered.h5ad",
    "halfbrain_yad_2_filtered.h5ad",
    "halfbrain_yad_3_filtered.h5ad",
    "halfbrain_yad_4_filtered.h5ad",
    "halfbrain_ac_1_filtered.h5ad",
    "halfbrain_ac_2_filtered.h5ad",
    "halfbrain_ac_3_filtered.h5ad",
    "halfbrain_ac_4_filtered.h5ad",
    "halfbrain_aad_1_filtered.h5ad",
    "halfbrain_aad_2_filtered.h5ad",
    "halfbrain_aad_3_filtered.h5ad",
    "halfbrain_aad_4_filtered.h5ad"
]
PIXEL_SIZE_UM = 60  # spatial resolution in microns

In [27]:

# -----------------------------
# Create output folder
# -----------------------------
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Output folder created/set: {OUTPUT_FOLDER}")

# -----------------------------
# Load mapping CSV
# -----------------------------
mapping_df = pd.read_csv(COMMON_MZS_CSV)

if not {"mzs", "common_group_name"}.issubset(mapping_df.columns):
    raise ValueError("CSV must contain 'mzs' and 'common_group_name'")

mz_to_common = dict(zip(mapping_df["mzs"].astype(str),
                        mapping_df["common_group_name"].astype(str)))

print(f"Loaded {len(mz_to_common)} common m/z mappings.")

# -----------------------------
# Process each MSI file
# -----------------------------
for fname in MSI_SAMPLE_FILES:
    fpath = os.path.join(MSI_INPUT_FOLDER, fname)
    print(f"\nProcessing: {fname}")

    if not os.path.exists(fpath):
        print(f"❌ File not found: {fpath}")
        continue

    adata = ad.read_h5ad(fpath)

    # ---- Keep only common m/z ----
    var_mzs = adata.var["mzs"].astype(str)
    mask = var_mzs.isin(mz_to_common.keys())
    kept = mask.sum()
    removed = len(mask) - kept
    print(f"✔ Keeping {kept} common features; removing {removed} others")

    adata = adata[:, mask].copy()

    # ---- Rename m/z based on common_group_name ----
    adata.var["mzs"] = [mz_to_common[mz] for mz in adata.var["mzs"].astype(str)]

    # ---- Shift x,y so the minimum becomes 0 ----
    if "x" in adata.obs and "y" in adata.obs:
        x0 = adata.obs["x"].min()
        y0 = adata.obs["y"].min()

        adata.obs["x"] = adata.obs["x"] - x0
        adata.obs["y"] = adata.obs["y"] - y0

        print(f"✔ Shifted x and y → start at zero (x0={x0}, y0={y0})")

        # ---- Convert to microns ----
        adata.obs["x_um"] = adata.obs["x"] * PIXEL_SIZE_UM
        adata.obs["y_um"] = adata.obs["y"] * PIXEL_SIZE_UM

        print("✔ Added x_um and y_um using 60 µm pixel size")

    else:
        print("⚠ Missing x or y → cannot shift coordinates or compute um")

    # ---- Save modified file ----
    outname = fname.replace(".h5ad", "_common.h5ad")
    outpath = os.path.join(OUTPUT_FOLDER, outname)
    adata.write_h5ad(outpath)

    print(f"💾 Saved: {outpath}")

print("\n=== FINISHED processing all MSI files ===")


Output folder created/set: /home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/
Loaded 2703 common m/z mappings.

Processing: halfbrain_yc_1_filtered.h5ad
✔ Keeping 528 common features; removing 472 others
✔ Shifted x and y → start at zero (x0=98, y0=5)
✔ Added x_um and y_um using 60 µm pixel size
💾 Saved: /home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/halfbrain_yc_1_filtered_common.h5ad

Processing: halfbrain_yc_2_filtered.h5ad
✔ Keeping 528 common features; removing 472 others
✔ Shifted x and y → start at zero (x0=86, y0=2)
✔ Added x_um and y_um using 60 µm pixel size
💾 Saved: /home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/halfbrain_yc_2_filtered_common.h5ad

Processing: halfbrain_yc_3_filtered.h5ad
✔ Keeping 528 common features; removing 472 others
✔ Shifted x and y → start at zero (x0=93, y0=2)
✔ Added x_um and y_um using 60 µm pixel size
💾 Saved: /ho